In [1]:
%pip install lightgbm
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

from xgboost import XGBRegressor

import joblib
import os

In [13]:
df = pd.read_csv("../data/processed/features_v2.csv")

print(df.shape)
print(df.columns)

(10908, 20)
Index(['raceId', 'driverId', 'constructorId', 'circuitId', 'year', 'date',
       'grid', 'positionOrder', 'points', 'avgLapTime_s', 'pitStopCount',
       'driver_form_avg', 'constructor_form_avg', 'driver_experience',
       'constructor_points_roll5', 'driver_circuit_avg_finish',
       'circuitType_street', 'era_ground_effect', 'era_hybrid', 'era_v8'],
      dtype='object')


In [14]:
target = "avgLapTime_s"

In [15]:
drop_cols = [
    "raceId",
    "driverId",
    "constructorId",
    "date",
    "positionOrder",
    "avgLapTime_s",
    "points"
]

features = [col for col in df.columns if col not in drop_cols]

print("Number of features:", len(features))

Number of features: 13


In [16]:
train = df[df["year"] < 2020]
test  = df[df["year"] >= 2020]

X_train = train[features]
X_test  = test[features]

y_train = train[target]
y_test  = test[target]

In [17]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

xgb = XGBRegressor(
    n_estimators=150,
    max_depth=4,
    random_state=42,
    n_jobs=-1
)

In [18]:
# Random Forest
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
print("RF R2:", r2_score(y_test, rf_preds))

RF R2: 0.12898988483199525


In [19]:
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
print("XGB R2:", r2_score(y_test, xgb_preds))

XGB R2: 0.12370881833179148


In [20]:
print(X_train.shape)

(8842, 13)


In [21]:
def evaluate(name, y_true, y_pred):
    print(f"\n{name}")
    print("MAE:", mean_absolute_error(y_true, y_pred))
    print("R2:", r2_score(y_true, y_pred))

evaluate("Random Forest", y_test, rf_preds)
evaluate("XGBoost", y_test, xgb_preds)

rf_r2 = r2_score(y_test, rf_preds)
xgb_r2 = r2_score(y_test, xgb_preds)

if rf_r2 >= xgb_r2:
    best_model = rf
    best_name = "random_forest"
    best_r2 = rf_r2
else:
    best_model = xgb
    best_name = "xgboost"
    best_r2 = xgb_r2

print(f"\nBest model selected: {best_name} (R2 = {best_r2:.4f})")


Random Forest
MAE: 13.235664562483244
R2: 0.12898988483199525

XGBoost
MAE: 13.163884173845451
R2: 0.12370881833179148

Best model selected: random_forest (R2 = 0.1290)


In [22]:
os.makedirs("models", exist_ok=True)

# Save model and feature list
joblib.dump(best_model, "../models/avg_lap_time_model.joblib")
joblib.dump(features, "../models/avg_lap_time_features.joblib")

print("Model and features saved successfully.")

Model and features saved successfully.
